# Olist Marketplace End-to-End Analytics 

## 1. Environment Setup & Library Imports
Initialize the project environment and import core libraries

In [1]:
import os
import pandas as pd

from pathlib import Path

##  Data Ingestion
Load the raw Olist relational datasets from local storage into Pandas DataFrames for preprocessing.

In [2]:
base_path = Path("/mnt/e/Sanjana_Code/ShopSphere/data/")

customers = pd.read_csv(f'{base_path}/raw/olist_customers_dataset.csv')
orders = pd.read_csv(f'{base_path}/raw/olist_orders_dataset.csv')
order_items = pd.read_csv(f'{base_path}/raw/olist_order_items_dataset.csv')
payments = pd.read_csv(f'{base_path}/raw/olist_order_payments_dataset.csv')
products = pd.read_csv(f'{base_path}/raw/olist_products_dataset.csv')
translations = pd.read_csv(f'{base_path}/raw/product_category_name_translation.csv')
reviews = pd.read_csv(f'{base_path}/raw/olist_order_reviews_dataset.csv')
sellers = pd.read_csv(f'{base_path}/raw/olist_sellers_dataset.csv')
geolocation = pd.read_csv(f'{base_path}/raw/olist_geolocation_dataset.csv')

Data Wrangling & Preprocessing

## Consolidation
Map dataset names to their respective DataFrames in a unified dictionary to streamline iterative operations and automated cleaning checks.

In [3]:
# Create a dictionary to map table names to their respective DataFrames
datasets = {
    'Customers': customers,
    'Orders': orders,
    'Order Items': order_items,
    'Payments': payments,
    'Products': products,
    'Translations': translations,
    'Reviews': reviews,
    'Sellers': sellers,
    'Geolocations': geolocation
}


## Duplicate Assessment & Removal
Iterate through all datasets to detect duplicate rows. Automatically remove identified duplicates in-place to ensure data uniqueness.

In [ ]:
# Check for duplicate rows in each dataset
for name, df in datasets.items():
    print(f"{name} Table has {df.shape[0]} rows and {df.shape[1]} columns.")
    duplicate_count = df.duplicated().sum()
    print(f"{name} Table has {duplicate_count} duplicate rows.")
    
    # If duplicates exist, drop them in-place
    if duplicate_count > 0:
        df.drop_duplicates(inplace=True)
        print(f"--> Fixed: Removed duplicates from {name} Table.")

Customers Table has (99441, 5).
Customers Table has 0 duplicate rows.
Orders Table has (99441, 8).
Orders Table has 0 duplicate rows.
Order Items Table has (112650, 7).
Order Items Table has 0 duplicate rows.
Payments Table has (103886, 5).
Payments Table has 0 duplicate rows.
Products Table has (32951, 9).
Products Table has 0 duplicate rows.
Translations Table has (71, 2).
Translations Table has 0 duplicate rows.
Reviews Table has (99224, 7).
Reviews Table has 0 duplicate rows.
Sellers Table has (3095, 4).
Sellers Table has 0 duplicate rows.
Geolocations Table has (1000163, 5).
Geolocations Table has 261831 duplicate rows.
--> Fixed: Removed duplicates from Geolocations Table.


## Missing Values Exploration
Scan all datasets to identify columns containing missing or null values. This assessment establishes the baseline for handling missing data across the relational schema.

In [5]:
# Loop through the dictionary and print .info() for each DataFrame
for name, df in datasets.items():
    null_counts = df.isnull().sum()
    # Filter to show only columns that have at least 1 null value
    missing_columns = null_counts[null_counts > 0]
    
    print(f"==================== {name.upper()} MISSING VALUES ====================")
    if not missing_columns.empty:
        print(missing_columns)
    else:
        print("No missing values found in this table.")
    print("\n" + "_"*60 + "\n")

==================== CUSTOMERS MISSING VALUES ====================
No missing values found in this table.

____________________________________________________________

==================== ORDERS MISSING VALUES ====================
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

____________________________________________________________

==================== ORDER ITEMS MISSING VALUES ====================
No missing values found in this table.

____________________________________________________________

==================== PAYMENTS MISSING VALUES ====================
No missing values found in this table.

____________________________________________________________

==================== PRODUCTS MISSING VALUES ====================
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
pr

> ### Missing Data Insights
> * **Clean Tables:** `Customers`, `Order Items`, `Payments`, `Translations`, and `Sellers` have **0** missing values.
> * **Orders Table:** Contains missing timestamps related to order fulfillment (`order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`).
> * **Products Table:** Missing metadata for 610 products and dimensions for 2 products.
> * **Reviews Table:** High volume of missing values in text fields (`review_comment_title`, `review_comment_message`), which is expected as comments are optional.

## Handling Missing Values: 



## Orders Table
### Decision: Retain Missing Timestamps as `NULL`
Missing values identified in fulfillment timestamps (`order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date`) will intentionally be left as **`NULL`**.

**Business Justification:**
* **Data Integrity:** Missing timestamps align with specific order statuses (e.g., `canceled` or `processing`), where the event naturally did not occur.
* **Downstream Analytics:** Imputing dummy dates would skew shipping duration, logistics performance, and delivery delay metrics during subsequent analysis and dashboarding 

## Products Table

### Decision: Fill Missing Values
* **Why?** We cannot delete these products because they are linked to real sales and revenue.
* **Action:** 
  * Fill missing `product_category_name` with **`'unknown'`**.
  * Fill missing numbers (weights, sizes, photo quantities) with **`0`** to keep data types consistent.


**Note:** For a detailed business analysis and logistics justification regarding why we used `0` instead of the Mean/Median, please check the **README.md** file.

In [6]:
# Fill categorical column with 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Fill numerical product specs with 0
num_product_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 
                    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in num_product_cols:
    products[col] = products[col].fillna(0)

# Verify that there are absolutely no missing values left in the entire products DataFrame
print("Total missing values in Products table:", products.isnull().sum().sum())


Total missing values in Products table: 0


#### Perfect no more NaNs in products table 
---

## Reviews Table

### Decision: Fill Missing Review Text
* **Why?** Customers rated the orders using stars (1 to 5) but chose not to write a title or a message. It is normal and we must keep the ratings.
* **Action:** 
  * Fill missing `review_comment_title` with **`'No Title'`**.
  * Fill missing `review_comment_message` with **`'No Message'`**.

In [7]:
# Fill text columns with standard placeholder text
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
# Corrected typo from 'lenght' to standard English naming conventions in comments if needed, but keeping actual column name
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Message')

# Verify that there are no missing values left in the review comment columns
print("Missing values in comment titles:", reviews['review_comment_title'].isnull().sum())
print("Missing values in comment messages:", reviews['review_comment_message'].isnull().sum())

Missing values in comment titles: 0
Missing values in comment messages: 0


---

# Cheacking and fixing Datatypes for each Table 

### orders Table:

In [8]:
# Check current data types for Orders table
print(orders.dtypes)

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


### Observation: Incorrect Date Formats
* **Issue:** All timestamp columns (e.g., `order_purchase_timestamp`, `order_approved_at`) are currently stored as text (`object`).
* **Impact:** We cannot perform time-based analysis or duration calculations in this format.

---


## Converting Timestamps to Datetime
* **Action:** Convert the identified date columns from text (`object`) to proper `datetime` format.
* **Handling Errors:** Used `errors='coerce'` to safely handle any formatting issues by setting faulty dates to `NaT` (Not a Time).

In [9]:
# List of columns that need to be converted to datetime
orders_date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# Convert columns to datetime
for col in orders_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Verify the changes for this table specifically
print("\n--- Verified Data Types for Orders Table ---")
print(orders[orders_date_cols].dtypes)


--- Verified Data Types for Orders Table ---
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [10]:
# Print the data types of the orders table after conversion
print(orders.dtypes)

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## Reviews Table

In [11]:
# Check current data types for Reviews table
print(reviews.dtypes)

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object


## Reviews Table

### Observation: Incorrect Date Formats
* **Issue:** The date columns `review_creation_date` and `review_answer_timestamp` are currently stored as text (`object`).
* **Impact:** These need to be converted to actual dates to track review response times accurately.
---

## Converting Reviews Timestamps to Datetime
* **Action:** Convert `review_creation_date` and `review_answer_timestamp` from text to proper `datetime` format.
* **Goal:** Enable future time-based tracking, such as calculating the response time for customer reviews.

In [12]:
# List of columns that need to be converted to datetime in Reviews
reviews_date_cols = ['review_creation_date', 'review_answer_timestamp']

# Convert columns to datetime with error handling
for col in reviews_date_cols:
    reviews[col] = pd.to_datetime(reviews[col], errors='coerce')

# Verify the changes for this table specifically
print("\n--- Verified Data Types for Reviews Table ---")
print(reviews[reviews_date_cols].dtypes)


--- Verified Data Types for Reviews Table ---
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [13]:
print(reviews.dtypes)

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


## Products Table 

In [14]:
# Check current data types for Products table
print(products.dtypes)

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


## Products Table

### Observation: Inconsistent Numeric Types
* **Issue:** Columns like `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` are stored as decimals (`float64`).
* **Impact:** These represent counts and lengths, so they logically should be whole numbers (`int64`) for clean representation.
---

### Fixing the datatypes 

In [15]:
# List of columns to convert from float to int
cols_to_int = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty']

# Convert using astype()
for col in cols_to_int:
    products[col] = products[col].astype('int64')

# Verify the changes
print("--- Verified Data Types for Products Table ---")
print(products[cols_to_int].dtypes)

--- Verified Data Types for Products Table ---
product_name_lenght           int64
product_description_lenght    int64
product_photos_qty            int64
dtype: object


In [16]:
print(products.dtypes)

product_id                        str
product_category_name             str
product_name_lenght             int64
product_description_lenght      int64
product_photos_qty              int64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


### All the columns dtypes are now fixed
---

## Data Type Assessment: Remaining Tables
* **Action:** Scan the data types for the remaining datasets (`Customers`, `Order Items`, `Payments`, `Sellers`, and `Translations`).
* **Goal:** Verify if there are any remaining formatting inconsistencies before finalizing the data type adjustment phase.

In [17]:
remaining_datasets = {
    'Customers': customers,
    'Order Items': order_items,
    'Payments': payments,
    'Sellers': sellers,
    'Translations': translations,
    'Geolocations': geolocation
}

for name, df in remaining_datasets.items():
    print(f"--- {name} Table Data Types ---")
    print(df.dtypes)
    print("\n" + "_"*40 + "\n")

--- Customers Table Data Types ---
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

________________________________________

--- Order Items Table Data Types ---
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

________________________________________

--- Payments Table Data Types ---
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

________________________________________

--- Sellers Table Data Types ---
seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object



## Observation from Remaining Tables
* **Issue:** Inside the `Order Items` table, the column `shipping_limit_date` is currently stored as text (`object`).
* **Conclusion:** All other columns across these datasets have correct data types (IDs and text as `object`, values as `float64`/`int64`) and look ready.
---

## Converting Shipping Limit Date to Datetime

In [18]:
# Convert shipping_limit_date to datetime in Order Items table
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')

# Verify the fix
print("Order Items Table - Shipping Limit Date Type:")
print(order_items['shipping_limit_date'].dtypes)

Order Items Table - Shipping Limit Date Type:
datetime64[us]


In [19]:
print(order_items.dtypes)

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object


### Storing Table Schema and final table

Idea is to use schema for table creation and clean data in Postgress

In [20]:
processed_path = Path(base_path) / "processed"
if not processed_path.exists():
    processed_path.mkdir(parents=True, exist_ok=True)

table_schema_path = Path(base_path) / "table_schema"
if not table_schema_path.exists():
    table_schema_path.mkdir(parents=True, exist_ok=True)

In [28]:
for name, df in datasets.items():
    # Save the DataFrame to a CSV file in the processed folder
    df.to_csv(f"{processed_path}/{name.lower().replace(' ', '_')}_processed.csv", index=False)
    
    # Save the schema (column names, data types, nullable) to a separate JSON file in the table_schema folder
    schema_df = pd.DataFrame(df.dtypes, columns=['Data Type']).reset_index()
    schema_df.rename(columns={'index': 'Column Name'}, inplace=True)
    schema_df['Data Type'] = schema_df['Data Type'].astype(str)
    schema_df['Nullable'] = schema_df['Column Name'].apply(lambda col: df[col].isnull().any())
    schema_df.to_json(f"{table_schema_path}/{name.lower().replace(' ', '_')}_schema.json", orient='records', index=False)